# Session 14 Lab: AI Agents
**Course:** Noob to AI Expert | **Track:** Expert

Build a complete agent loop with tool calling, memory, and a multi-agent pipeline.

In [ ]:
!pip install anthropic -q
print('✅ Ready')

In [ ]:
import anthropic, json, datetime, os
os.environ.setdefault('ANTHROPIC_API_KEY', 'your-key-here')
client = anthropic.Anthropic()

tools = [
    {
        'name': 'search_web',
        'description': 'Search the web for current information.',
        'input_schema': {'type': 'object', 'properties': {'query': {'type': 'string'}}, 'required': ['query']}
    },
    {
        'name': 'calculate',
        'description': 'Evaluate a math expression.',
        'input_schema': {'type': 'object', 'properties': {'expression': {'type': 'string'}}, 'required': ['expression']}
    },
    {
        'name': 'get_date',
        'description': 'Get the current date and time.',
        'input_schema': {'type': 'object', 'properties': {}}
    }
]

def execute_tool(name, inputs):
    if name == 'search_web':
        return f'[Mock search results for "{inputs["query"]}"] GDP of Germany in 2023 was approximately $4.08 trillion USD.'
    if name == 'calculate':
        try: return str(eval(inputs['expression'], {'__builtins__': {}}))
        except: return 'Error evaluating expression'
    if name == 'get_date':
        return datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    return 'Tool not found'

print('Tools defined:', [t['name'] for t in tools])

In [ ]:
def run_agent(user_message, max_turns=10):
    messages = [{'role': 'user', 'content': user_message}]
    turn = 0

    while turn < max_turns:
        turn += 1
        resp = client.messages.create(
            model='claude-haiku-4-5-20251001', max_tokens=1024,
            tools=tools, messages=messages
        )

        if resp.stop_reason == 'end_turn':
            for block in resp.content:
                if hasattr(block, 'text'): return block.text

        if resp.stop_reason == 'tool_use':
            messages.append({'role': 'assistant', 'content': resp.content})
            results = []
            for block in resp.content:
                if block.type == 'tool_use':
                    result = execute_tool(block.name, block.input)
                    print(f'  [Tool: {block.name}] → {result[:80]}')
                    results.append({'type': 'tool_result', 'tool_use_id': block.id, 'content': result})
            messages.append({'role': 'user', 'content': results})

    return 'Max turns reached'

result = run_agent("What's today's date? Also, what is 15% of Germany's GDP?")
print('\nFinal answer:', result)